# Training 08 · Fresh attackers against every predictor (E1)

For each predictor (R0, R1–R3, R3 multi, the three baselines; 3 seeds) a new EnCodec attacker and a new DAC
attacker are trained with GRPO (2000 steps) and then evaluated together with waveform PGD, the replayed round-1
attacker and random directions, exactly like the original job (about 30 min each). The evaluation-only counterpart,
which reuses the released attackers, is `tables/table2_loop_audit.ipynb` (Table II).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1']]
print(len(KEYS), "runs:", ", ".join(KEYS))

24 runs: e1_R0_s0, e1_R3_s0, e1_R0_s1, e1_R3_s1, e1_R0_s2, e1_R3_s2, e1_baseline_noise_s0, e1_baseline_pgd_s0, e1_baseline_random_s0, e1_baseline_noise_s1, e1_baseline_pgd_s1, e1_baseline_random_s1, e1_baseline_noise_s2, e1_baseline_pgd_s2, e1_baseline_random_s2, e1_R1_s0, e1_R2_s0, e1_R1_s1, e1_R2_s1, e1_R1_s2, e1_R2_s2, e1_R3ms_s0, e1_R3ms_s1, e1_R3ms_s2


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: e1_attack
task: attack_eval
phase: main
seed: 0
run_name: e1_attack_R0_s0__2026-09-15_16-19-29
target_name: R0
target_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
transfer_attacker_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/hnp_s0.round1.attacker.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
e1:
  eval_set: bvcc/test
  max_utts: null
  judges:
  - utmos
  - dnsmos
attackers:
- name: grpo_encodec
  type: grpo
- name: grpo_dac
  type: grpo
  attack_overrides:
    codec:
      name: dac_16khz
      quantize: true
- name: pgd_wave
  type: pgd
  pgd:
    eps: 0.002
    steps: 50
    step_size: 0.0002
    batch_size: 8
- name: random_encodec
  type: random
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for e1_R0_s0 ->


#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_e1_R0_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key e1_R0_s0



## Retrained vs original

In [6]:
import json
released = repro.fetch_results()
rows = {}
for k in KEYS:
    for src, d in (("original", released / Path(reg[k]["run_dir"]).name), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "results/e1_summary.json"
        if f.exists():
            rows[(k, src)] = pd.json_normalize(json.loads(f.read_text())).iloc[0]
df = pd.DataFrame(rows).T
df[[c for c in df.columns if c.endswith((".asr", ".mean_gain"))]] if len(df) else "no runs found"

,,attackers.grpo_encodec.asr,attackers.grpo_encodec.mean_gain,attackers.grpo_dac.asr,attackers.grpo_dac.mean_gain,attackers.pgd_wave.asr,attackers.pgd_wave.mean_gain,attackers.random_encodec.asr,attackers.random_encodec.mean_gain,attackers.transfer_r0_round1.asr,attackers.transfer_r0_round1.mean_gain
e1_R0_s0,original,0.205441,0.110306,0.015009,-0.038491,0.997186,1.707552,0.005629,-0.50184,0.212946,0.127455
e1_R3_s0,original,0.062852,-0.107807,0.288931,0.192381,1.0,2.026604,0.0,-2.554409,0.0,-3.466236
e1_R0_s1,original,0.030019,-0.162436,0.341463,0.202847,1.0,1.875572,0.017824,-0.392318,0.232645,0.139203
e1_R3_s1,original,0.025328,-0.076139,0.269231,0.170483,1.0,2.183,0.0,-1.891607,0.0,-3.031543
e1_R0_s2,original,0.307692,0.175789,0.003752,-0.100485,1.0,1.77428,0.012195,-0.422058,0.307692,0.182428
e1_R3_s2,original,0.060976,0.034922,0.320826,0.245126,1.0,2.097143,0.0,-1.193605,0.0,-1.77214
e1_baseline_noise_s0,original,0.273921,0.145513,0.375235,0.25354,1.0,1.988274,0.013133,-0.449717,0.243902,0.152791
e1_baseline_pgd_s0,original,0.190432,0.136631,0.395872,0.276091,0.9803,1.665154,0.012195,-0.747897,0.256098,0.173158
e1_baseline_random_s0,original,0.086304,0.098814,0.409944,0.288681,1.0,2.029967,0.0,-2.087249,0.002814,-1.064405
e1_baseline_noise_s1,original,0.168856,0.125147,0.324578,0.209948,1.0,2.106348,0.013133,-0.464631,0.17636,0.078711
